<a href="https://colab.research.google.com/github/MahathiKadiyala/24B11DS084_DriveReady_OPC_2026/blob/main/Day-4/Agentic_AI_Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q langchain langchain-groq gspread pydantic


In [14]:

from google.colab import userdata
from langchain_groq import ChatGroq

# Retrieve API key from Colab Secrets
GROQ_API_KEY = userdata.get("agentic_ai_GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Groq API key not found. Check Colab Secrets.")

# Initialize Groq model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3,
    api_key=GROQ_API_KEY
)

print("Groq model initialized successfully!")


Groq model initialized successfully!


In [15]:

response = llm.invoke(
    "Introduce yourself as an AI assistant in one sentence."
)

print(response.content)


I’m ChatGPT, your friendly AI assistant ready to help you with information, ideas, and creative solutions in a single sentence.


In [16]:

from langchain.tools import tool

# Tool 1: Get estimated service price
@tool
def get_service_price(service_type: str) -> str:
    """
    Get the estimated price of an EV maintenance service.
    Use this tool when a customer asks about service charges.
    """

    prices = {
        "general_service": 1500,
        "battery_check": 800,
        "tyre_check": 300
    }

    service_type = service_type.lower()

    if service_type in prices:
        return f"Estimated service price: Rs {prices[service_type]}"

    return "Price not available. Please contact the service team."


# Tool 2: Check supported services
@tool
def check_service_supported(service_type: str) -> str:
    """
    Check whether the service centre supports a particular EV service.
    This checks service support, not appointment slot availability.
    """

    available_services = [
        "general_service",
        "battery_check",
        "tyre_check"
    ]

    if service_type.lower() in available_services:
        return f"{service_type} is supported by our service centre."

    return f"{service_type} is currently not supported by our service centre."


print("VoltCare AI tools created successfully!")


VoltCare AI tools created successfully!


In [17]:

print(
    get_service_price.invoke({
        "service_type": "battery_check"
    })
)

print(
    check_service_supported.invoke({
        "service_type": "general_service"
    })
)

print(
    check_service_supported.invoke({
        "service_type": "car_washing"
    })
)


Estimated service price: Rs 800
general_service is supported by our service centre.
car_washing is currently not supported by our service centre.


In [18]:

from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are VoltCare AI, an EV customer and service assistant.

Your responsibilities:
1. Help customers with EV maintenance enquiries.
2. Use get_service_price to retrieve estimated service prices.
3. Use check_service_supported to check supported services.
4. Inform customers when a service is not supported.
5. Never invent prices or company information.
6. Do not claim that an appointment is booked.
7. Be polite, clear and helpful.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_service_price,
        check_service_supported
    ],
    system_prompt=SYSTEM_PROMPT
)

print("VoltCare AI agent created successfully!")


VoltCare AI agent created successfully!


In [19]:

questions = [
    "What is the price of a battery check?",
    "Do you provide general EV servicing?",
    "Do you provide car washing?"
]

for question in questions:
    print("\nCustomer:", question)

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    print("VoltCare AI:", result["messages"][-1].content)



Customer: What is the price of a battery check?
VoltCare AI: I’m sorry, but I don’t have a price listed for a battery check at the moment. Please reach out to our service team directly and they’ll be happy to give you an accurate estimate. If there’s anything else I can help with, just let me know!

Customer: Do you provide general EV servicing?
VoltCare AI: I’m sorry, but our service centre does not currently offer general EV servicing. If you have a specific maintenance need—such as battery checks, software updates, or component replacements—let me know and I can check if that service is available and provide an estimated price.

Customer: Do you provide car washing?
VoltCare AI: I’m sorry, but our service centre does not offer car washing at this time. If you have any other maintenance or service questions, feel free to let me know!


In [20]:

def ask_voltcare(question):
    try:
        result = agent.invoke({
            "messages": [
                {"role": "user", "content": question}
            ]
        })

        return result["messages"][-1].content

    except Exception as e:
        error_message = str(e).lower()

        if "429" in error_message or "rate_limit" in error_message:
            return "The service is temporarily busy. Please try again shortly."

        return "Sorry, something went wrong. Please try again."


print(ask_voltcare("What is the price of a tyre check?"))


I’m sorry, but I don’t have a price listed for a tyre check at the moment. Please get in touch with our service team for the most up‑to‑date pricing and any additional details you might need. If there’s anything else I can help you with, just let me know!


In [21]:

from pydantic import BaseModel, Field

class EVServiceEnquiry(BaseModel):
    customer_name: str | None = Field(
        default=None,
        description="Customer's name"
    )

    vehicle_model: str | None = Field(
        default=None,
        description="Customer's electric vehicle model"
    )

    service_type: str = Field(
        description="Requested EV maintenance service"
    )

    contact_number: str | None = Field(
        default=None,
        description="Customer's contact number"
    )

    preferred_date: str | None = Field(
        default=None,
        description="Customer's preferred service date"
    )

    issue_description: str | None = Field(
        default=None,
        description="Description of the customer's issue"
    )


print("EVServiceEnquiry model created successfully!")


EVServiceEnquiry model created successfully!


In [22]:

structured_llm = llm.with_structured_output(EVServiceEnquiry)

customer_messages = [
    "I am Rahul. I own a Tata Nexon EV and need a battery check next Monday.",

    "My name is Priya. I have an MG ZS EV. I want general servicing.",

    "This is Arjun. I drive an electric scooter. I need a tyre check. My number is 9876543210.",

    "I am Sneha. I own a Hyundai Kona EV. I am facing a battery-related issue and want it checked tomorrow.",

    "My name is Kiran. I have a Tata Tiago EV. I need general service. Please contact me on 9123456780."
]

extracted_enquiries = []

for message in customer_messages:
    print("\nCustomer message:", message)

    try:
        enquiry = structured_llm.invoke(
            "Extract the EV service enquiry details from this message. "
            "Do not invent missing information. Keep relative dates "
            "as expressed by the customer.\n\n" + message
        )

        extracted_enquiries.append(enquiry)

        print("Structured output:")
        print(enquiry.model_dump())

    except Exception as e:
        print("Extraction failed:", e)

print("\nSuccessfully extracted:", len(extracted_enquiries))



Customer message: I am Rahul. I own a Tata Nexon EV and need a battery check next Monday.
Structured output:
{'customer_name': 'Rahul', 'vehicle_model': 'Tata Nexon EV', 'service_type': 'battery check', 'contact_number': None, 'preferred_date': 'next Monday', 'issue_description': None}

Customer message: My name is Priya. I have an MG ZS EV. I want general servicing.
Structured output:
{'customer_name': 'Priya', 'vehicle_model': 'MG ZS EV', 'service_type': 'general servicing', 'contact_number': None, 'preferred_date': None, 'issue_description': 'general servicing'}

Customer message: This is Arjun. I drive an electric scooter. I need a tyre check. My number is 9876543210.
Structured output:
{'customer_name': 'Arjun', 'vehicle_model': 'electric scooter', 'service_type': 'tyre check', 'contact_number': '9876543210', 'preferred_date': None, 'issue_description': 'tyre check'}

Customer message: I am Sneha. I own a Hyundai Kona EV. I am facing a battery-related issue and want it checked to

In [23]:

# Temporary storage for customer enquiries
service_enquiries = []

print("Temporary enquiry storage initialized!")


Temporary enquiry storage initialized!


In [36]:

@tool
def log_service_enquiry(
    customer_name: str,
    vehicle_model: str,
    service_type: str,
    issue_description: str,
    contact_number: str | None = None,
    preferred_date: str | None = None
) -> str:
    """
    Record a customer's EV service enquiry
    permanently in Google Sheets.
    """

    try:
        enquiry = EVServiceEnquiry(
            customer_name=customer_name,
            vehicle_model=vehicle_model,
            service_type=service_type,
            issue_description=issue_description,
            contact_number=contact_number,
            preferred_date=preferred_date
        )

        data = enquiry.model_dump()

        worksheet.append_row([
            data["customer_name"] or "",
            data["vehicle_model"] or "",
            data["service_type"],
            data["issue_description"] or "",
            data["contact_number"] or "",
            data["preferred_date"] or ""
        ])

        return (
            f"Service enquiry for {customer_name} "
            "saved successfully in Google Sheets."
        )

    except Exception as e:
        return f"Unable to save enquiry: {str(e)}"


print("Google Sheets enquiry logging tool created successfully!")



Google Sheets enquiry logging tool created successfully!


In [37]:

voltcare_agent = create_agent(
    model=llm,
    tools=[
        get_service_price,
        check_service_supported,
        log_service_enquiry
    ],
    system_prompt="""
    You are VoltCare AI, an EV customer and service assistant.

    Your responsibilities:
    1. Answer EV maintenance-related questions.
    2. Use tools to check estimated service prices.
    3. Check whether a service is supported.
    4. Collect customer and vehicle details when a customer
       wants to submit a service enquiry.
    5. Ask for missing required details before recording.
    6. Use log_service_enquiry to record the enquiry.
    7. Never invent prices or company information.
    8. Never claim an appointment is booked.
    9. Confirm that an enquiry was recorded only after
       the logging tool succeeds.
    10. Be polite and helpful.
    """
)

print("Complete VoltCare AI agent created successfully!")


Complete VoltCare AI agent created successfully!


In [38]:

customer_query = """
My name is Rahul.
I own a Tata Nexon EV.
I need a battery check.
My contact number is 9876543210.
I prefer next Monday.
Please record my service enquiry.
"""

result = voltcare_agent.invoke({
    "messages": [
        {"role": "user", "content": customer_query}
    ]
})

print("VoltCare AI:")
print(result["messages"][-1].content)


VoltCare AI:
✅ Your service enquiry has been recorded successfully.  
We’ve logged the following details:

- **Name:** Rahul  
- **Vehicle:** Tata Nexon EV  
- **Service requested:** Battery check  
- **Preferred date:** Next Monday  
- **Contact number:** 987 654 3210  

We’ll keep you posted on the next steps. If you have any other questions or need further assistance, just let us know!


In [39]:

print("Total enquiries saved:",
      len(worksheet.get_all_records()))

for enquiry in worksheet.get_all_records():
    print(enquiry)
    print("-" * 40)


Total enquiries saved: 2
{'Customer Name': 'Rahul', 'EV Model': 'Tata Nexon EV', 'Issue': 'Battery charging problem', 'Urgency': 'High', 'Contact': 'TEST-9876543210', '': ''}
----------------------------------------
{'Customer Name': 'Rahul', 'EV Model': 'Tata Nexon EV', 'Issue': 'battery check', 'Urgency': 'battery check', 'Contact': 9876543210, '': 'next Monday'}
----------------------------------------


In [28]:

@tool("tool1")
def tool1(service_type: str) -> str:
    """does stuff"""

    return "This service is supported."


test_agent = create_agent(
    model=llm,
    tools=[get_service_price, tool1],
    system_prompt="""
    You are an EV service assistant.
    Use the available tools when appropriate.
    """
)

test_result = test_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Is battery checking supported?"
        }
    ]
})

print(test_result["messages"][-1].content)


Yes, battery health checks are part of our standard maintenance services. If you’d like to schedule a battery inspection or want to know the cost, just let me know!


In [29]:

trace_result = voltcare_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of a battery check?"
        }
    ]
})

messages = trace_result["messages"]

# Count AI messages
model_calls = sum(
    1 for message in messages
    if message.type == "ai"
)

# Count tool calls
tool_calls = sum(
    len(message.tool_calls)
    for message in messages
    if message.type == "ai"
)

# Count tool results
tool_results = sum(
    1 for message in messages
    if message.type == "tool"
)

print("AI messages:", model_calls)
print("Tool calls:", tool_calls)
print("Tool results:", tool_results)

print("\nAgent trace:")

for message in messages:
    print("\nMessage type:", message.type)
    print("Content:", message.content)

    if message.type == "ai" and message.tool_calls:
        print("Tool calls:", message.tool_calls)


AI messages: 2
Tool calls: 1
Tool results: 1

Agent trace:

Message type: human
Content: What is the price of a battery check?

Message type: ai
Content: 
Tool calls: [{'name': 'get_service_price', 'args': {'service_type': 'battery check'}, 'id': 'fc_11ccb35c-1569-40b0-8fab-2a3a217c5ef5', 'type': 'tool_call'}]

Message type: tool
Content: Price not available. Please contact the service team.

Message type: ai
Content: I’m sorry, but I don’t have a price listed for a battery check at the moment. It would be best to get in touch with our service team directly—they can give you an accurate quote and help schedule an appointment. If you’d like, I can log a service enquiry for you so you can get a quick response. Just let me know your name, vehicle model, the issue you’re experiencing, and a preferred date if you have one.


In [30]:

voltcare_facts = [
    "VoltCare AI is an EV customer and service assistance prototype.",
    "It helps customers with EV maintenance enquiries.",
    "The sample service catalogue includes general servicing.",
    "The sample service catalogue includes battery checks.",
    "The sample service catalogue includes tyre checks.",
    "The sample estimated general service price is Rs 1500.",
    "The sample estimated battery check price is Rs 800.",
    "The sample estimated tyre check price is Rs 300.",
    "All sample prices are illustrative, not verified company quotations.",
    "Customers can enquire about supported services.",
    "Customers can provide their vehicle model.",
    "Customers can provide their preferred service date.",
    "Customers can provide their contact number.",
    "The prototype can record customer service enquiries.",
    "Pydantic validates customer enquiry information.",
    "LangChain tools allow the agent to perform service operations.",
    "The agent should not invent company information.",
    "The agent should not claim an appointment is booked.",
    "Appointment booking requires a real scheduling system.",
    "VoltCare AI aims to simplify EV service interactions."
]

with open("voltcare_business_facts.txt", "w") as file:
    file.write("\n".join(voltcare_facts))

print("Business facts prepared successfully!")
print("Total facts:", len(voltcare_facts))


Business facts prepared successfully!
Total facts: 20


In [31]:
!pip install -q gspread google-auth

In [32]:
from google.colab import auth

auth.authenticate_user()

In [33]:
import google.auth
import gspread

# Get authenticated Google credentials
creds, project = google.auth.default()

# Authorize gspread
gc = gspread.authorize(creds)

print("Google Sheets connected successfully!")

Google Sheets connected successfully!


In [34]:
SPREADSHEET_ID = "13ercP1O_lDqLCqzsz2i3Qp5_AUppg1a9v9a065X5DnM"

sheet = gc.open_by_key(SPREADSHEET_ID)

worksheet = sheet.sheet1

print("Connected to:", sheet.title)

Connected to: VoltCare_AI_Enquiries


In [35]:
test_enquiry = [
    "Rahul",
    "Tata Nexon EV",
    "Battery charging problem",
    "High",
    "TEST-9876543210"
]

worksheet.append_row(test_enquiry)

print("Enquiry saved successfully!")

Enquiry saved successfully!
